# Lung Disease Classification — 3-Class (Normal / Pneumonia / Tuberculosis)

**Key design decisions:**
- `normal` = equal mix from both datasets (pneumonia-dataset Normal + TB-dataset Normal)
- `pneumonia` = pneumonia dataset PNEUMONIA folders (train+test+val merged)
- `tuberculosis` = TB dataset Tuberculosis folder
- **Upsampling**: every class guaranteed ≥ 1000 images via PIL augmentation
- **Zero leakage**: path-level split BEFORE any file copy; unique prefixed filenames prevent collisions
- **Split**: 80 / 10 / 10 stratified train/val/test

In [1]:
# ============================================================
# CELL 1 — Imports & Global Config
# ============================================================
import os, gc, json, time, shutil, random, hashlib, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageFilter

import torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from torchvision import transforms, models
from torchvision.models import (
    ResNet18_Weights, MobileNet_V3_Small_Weights, EfficientNet_B0_Weights,
    VGG16_Weights, DenseNet121_Weights, ResNet34_Weights, ResNet50_Weights,
    EfficientNet_V2_S_Weights, MobileNet_V3_Large_Weights, ConvNeXt_Tiny_Weights
)
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_recall_fscore_support
)
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# ── Paths ────────────────────────────────────────────────────
BASE_INPUT  = Path('/kaggle/input')
WORKING_DIR = Path('/kaggle/working')
DATASET_DIR = WORKING_DIR / 'lung_dataset'          # clean split dataset
ARTIFACT_DIR= WORKING_DIR / 'lung_artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}

# ── Hyperparameters ──────────────────────────────────────────
TARGET_PER_CLASS = 1200   # upsample/downsample all classes to this exact count
                           # (must be >= 1000 per requirement)
NORMAL_FROM_EACH = TARGET_PER_CLASS // 2  # 600 from pneu-dataset + 600 from TB-dataset

IMG_SIZE     = 224
BATCH_SIZE   = 16
EPOCHS       = 20
PATIENCE     = 4
LR           = 1e-4
USE_PRETRAINED = True

NUM_CLASSES  = 3
CLASS_NAMES  = ['normal', 'pneumonia', 'tuberculosis']

MODELS_TO_RUN = [
    'multinet', 'resnet18', 'mobilenet_v3_small', 'efficientnet_b0',
    'vgg16', 'densenet121', 'resnet34', 'resnet50',
    'efficientnet_v2_s', 'mobilenet_v3_large', 'convnext_tiny',
]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'Device : {device}  ({gpu_name})')
print(f'Target images per class : {TARGET_PER_CLASS}')
print(f'Normal split per source : {NORMAL_FROM_EACH}')

Device : cuda  (Tesla T4)
Target images per class : 1200
Normal split per source : 600


In [4]:
# ============================================================
# CELL 2 — Explore Input Structure
# ============================================================
print('Kaggle input directories:')
for p in sorted(BASE_INPUT.rglob('*')):
    if p.is_dir():
        print(' ', p)

Kaggle input directories:
  /kaggle/input/datasets
  /kaggle/input/datasets/paultimothymooney
  /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia
  /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray
  /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/__MACOSX
  /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/__MACOSX/chest_xray
  /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/__MACOSX/chest_xray/test
  /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/__MACOSX/chest_xray/test/NORMAL
  /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/__MACOSX/chest_xray/test/PNEUMONIA
  /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/__MACOSX/chest_xray/train
  /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/__MACOSX/chest_xray/train/NORMAL
  /kaggle/input/datasets/paultimothymooney/chest-xray-pneumoni

In [6]:
# ============================================================
# CELL 3 — Auto-locate Dataset Roots
# ============================================================
def find_dir(base, keywords):
    for p in sorted(base.rglob('*')):
        if p.is_dir():
            lp = str(p).lower()
            if all(k.lower() in lp for k in keywords):
                return p
    return None

PNEUMONIA_ROOT = find_dir(BASE_INPUT, ['chest_xray'])
if PNEUMONIA_ROOT is None:
    PNEUMONIA_ROOT = find_dir(BASE_INPUT, ['chest', 'xray'])

TB_ROOT = find_dir(BASE_INPUT, ['tb_chest_radiography_database'])
if TB_ROOT is None:
    TB_ROOT = find_dir(BASE_INPUT, ['tuberculosis'])

print('Pneumonia root :', PNEUMONIA_ROOT)
print('TB root        :', TB_ROOT)
assert PNEUMONIA_ROOT is not None, "Could not find chest_xray dataset folder!"
assert TB_ROOT is not None, "Could not find TB dataset folder!" 

Pneumonia root : /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray
TB root        : /kaggle/input/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset/TB_Chest_Radiography_Database


In [7]:
# ============================================================
# CELL 4 — Helpers
# ============================================================
def is_valid(path: Path) -> bool:
    """True if file is a real image (not macOS ._* junk)."""
    if path.name.startswith('._'):
        return False
    if path.suffix.lower() not in IMAGE_EXTENSIONS:
        return False
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False

def collect_valid(folders) -> list:
    """Collect all valid image paths from folder(s). Skips ._* and corrupt files."""
    if isinstance(folders, (str, Path)):
        folders = [folders]
    paths = []
    for folder in folders:
        folder = Path(folder)
        if not folder.exists():
            print(f'  [WARN] folder not found: {folder}')
            continue
        for f in folder.rglob('*'):
            if f.suffix.lower() in IMAGE_EXTENSIONS and not f.name.startswith('._'):
                paths.append(f)
    return paths

def md5_hash(path: Path) -> str:
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

def deduplicate(paths: list) -> list:
    seen, unique = set(), []
    for p in paths:
        try:
            h = md5_hash(p)
            if h not in seen:
                seen.add(h); unique.append(p)
        except Exception:
            pass
    return unique

print('Helpers defined.')

Helpers defined.


In [8]:
# ============================================================
# CELL 5 — Collect & Deduplicate Raw Images per Class
# ============================================================
# normal_pneu  = all NORMAL folders inside chest_xray (train+test+val)
# normal_tb    = TB dataset Normal folder
# pneumonia    = all PNEUMONIA folders inside chest_xray (train+test+val)
# tb           = TB dataset Tuberculosis folder

normal_pneu_dirs = list(PNEUMONIA_ROOT.rglob('NORMAL'))
normal_tb_dirs   = list(TB_ROOT.rglob('Normal'))
pneumonia_dirs   = list(PNEUMONIA_ROOT.rglob('PNEUMONIA'))
tb_dirs          = list(TB_ROOT.rglob('Tuberculosis'))

print('Dirs found:')
print('  normal (pneu dataset):', normal_pneu_dirs)
print('  normal (TB dataset)  :', normal_tb_dirs)
print('  pneumonia dirs       :', pneumonia_dirs)
print('  tuberculosis dirs    :', tb_dirs)

raw_normal_pneu = deduplicate(collect_valid(normal_pneu_dirs))
raw_normal_tb   = deduplicate(collect_valid(normal_tb_dirs))
raw_pneumonia   = deduplicate(collect_valid(pneumonia_dirs))
raw_tb          = deduplicate(collect_valid(tb_dirs))

print(f'\nAfter dedup:')
print(f'  normal (pneu source) : {len(raw_normal_pneu)}')
print(f'  normal (TB source)   : {len(raw_normal_tb)}')
print(f'  pneumonia            : {len(raw_pneumonia)}')
print(f'  tuberculosis         : {len(raw_tb)}')

assert len(raw_normal_pneu) >= 1, "No normal images from pneumonia dataset!"
assert len(raw_normal_tb)   >= 1, "No normal images from TB dataset!"
assert len(raw_pneumonia)   >= 1, "No pneumonia images found!"
assert len(raw_tb)          >= 1, "No TB images found!" 

Dirs found:
  normal (pneu dataset): [PosixPath('/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/val/NORMAL'), PosixPath('/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/test/NORMAL'), PosixPath('/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/train/NORMAL'), PosixPath('/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/val/NORMAL'), PosixPath('/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/test/NORMAL'), PosixPath('/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/train/NORMAL'), PosixPath('/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/__MACOSX/chest_xray/val/NORMAL'), PosixPath('/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/__MACOSX/chest_xray/test/NORMAL'), PosixPath('/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/__MACOSX/chest

In [9]:
# ============================================================
# CELL 6 — PIL Augmentation Upsampler
# ============================================================
# Used when a source has fewer images than needed.
# Generates augmented variants ON-THE-FLY from existing images.
# These are ONLY written to /train — val/test always use real images.

import random as _random
from pathlib import Path
from PIL import Image, ImageEnhance, ImageFilter

def pil_augment(img: Image.Image) -> Image.Image:
    """Apply random augmentations to a PIL image."""
    ops = []
    if _random.random() > 0.5:
        ops.append(lambda x: x.transpose(Image.FLIP_LEFT_RIGHT))
    if _random.random() > 0.5:
        angle = _random.uniform(-15, 15)
        ops.append(lambda x, a=angle: x.rotate(a, expand=False, fillcolor=(128,128,128)))
    if _random.random() > 0.5:
        factor = _random.uniform(0.8, 1.2)
        ops.append(lambda x, f=factor: ImageEnhance.Brightness(x).enhance(f))
    if _random.random() > 0.5:
        factor = _random.uniform(0.8, 1.2)
        ops.append(lambda x, f=factor: ImageEnhance.Contrast(x).enhance(f))
    if _random.random() > 0.3:
        ops.append(lambda x: x.filter(ImageFilter.GaussianBlur(radius=_random.uniform(0, 0.8))))
    for op in ops:
        img = op(img)
    return img

def upsample_paths(paths: list, target: int, cls_tag: str, split_tag: str, dst_dir: Path) -> list:
    """
    Copy/augment images from `paths` until `target` count reached.
    Files saved to dst_dir with unique prefix → zero name collisions.
    Returns list of saved file paths.
    """
    dst_dir.mkdir(parents=True, exist_ok=True)
    saved = []
    n = len(paths)
    idx = 0
    aug_count = 0

    for i, src in enumerate(paths):
        # Base copy (no augmentation)
        dst = dst_dir / f'{cls_tag}_{split_tag}_{i:05d}_orig{src.suffix.lower()}'
        try:
            img = Image.open(src).convert('RGB')
            img.save(dst)
            saved.append(dst)
        except Exception:
            continue

    # Augment until target reached
    while len(saved) < target:
        src = paths[aug_count % n]
        aug_count += 1
        dst = dst_dir / f'{cls_tag}_{split_tag}_{aug_count:06d}_aug.jpg'
        try:
            img = Image.open(src).convert('RGB')
            img = pil_augment(img)
            img.save(dst, quality=90)
            saved.append(dst)
        except Exception:
            pass
        if aug_count > target * 5:
            print(f'  [WARN] Could not reach target for {cls_tag}/{split_tag}')
            break

    return saved[:target]

print('Upsampler defined.')

Upsampler defined.


In [10]:
# ============================================================
# CELL 7 — Stratified Split BEFORE Copying
# ============================================================
# CRITICAL: We split the SOURCE path lists first,
# then copy/augment into dst folders.
# This guarantees zero leakage because augmented images
# in train are derived only from train-assigned source images.

random.shuffle(raw_normal_pneu)
random.shuffle(raw_normal_tb)
random.shuffle(raw_pneumonia)
random.shuffle(raw_tb)

def split_paths(paths, seed=SEED):
    """80/10/10 stratified split. Returns (train, val, test) lists."""
    train, temp = train_test_split(paths, test_size=0.20, random_state=seed)
    val, test   = train_test_split(temp,  test_size=0.50, random_state=seed)
    return train, val, test

# ── Split each source independently ──────────────────────────
npneu_train, npneu_val, npneu_test = split_paths(raw_normal_pneu)
ntb_train,   ntb_val,   ntb_test   = split_paths(raw_normal_tb)
pneu_train,  pneu_val,  pneu_test  = split_paths(raw_pneumonia)
tb_train,    tb_val,    tb_test    = split_paths(raw_tb)

print('Source split sizes (before balancing):')
print(f'  normal_pneu  : train={len(npneu_train)}  val={len(npneu_val)}  test={len(npneu_test)}')
print(f'  normal_tb    : train={len(ntb_train)}    val={len(ntb_val)}    test={len(ntb_test)}')
print(f'  pneumonia    : train={len(pneu_train)}   val={len(pneu_val)}   test={len(pneu_test)}')
print(f'  tuberculosis : train={len(tb_train)}     val={len(tb_val)}     test={len(tb_test)}')

# ── Leakage pre-check: verify no path appears in >1 split ────
for src_name, (tr, va, te) in [
    ('normal_pneu', (npneu_train, npneu_val, npneu_test)),
    ('normal_tb',   (ntb_train,   ntb_val,   ntb_test)),
    ('pneumonia',   (pneu_train,  pneu_val,  pneu_test)),
    ('tuberculosis',(tb_train,    tb_val,    tb_test)),
]:
    s_tr = set(tr); s_va = set(va); s_te = set(te)
    assert len(s_tr & s_va) == 0, f'{src_name}: train∩val overlap!'
    assert len(s_tr & s_te) == 0, f'{src_name}: train∩test overlap!'
    assert len(s_va & s_te) == 0, f'{src_name}: val∩test overlap!'
print('\n✓ Path-level leakage check passed.')

Source split sizes (before balancing):
  normal_pneu  : train=1263  val=158  test=158
  normal_tb    : train=2800    val=350    test=350
  pneumonia    : train=3396   val=424   test=425
  tuberculosis : train=557     val=70     test=70

✓ Path-level leakage check passed.


In [11]:
# ============================================================
# CELL 8 — Build Dataset Folders
# ============================================================
# Strategy per split:
#   TRAIN   → upsample to TARGET_PER_CLASS (augmentation allowed)
#   VAL     → use real images only (min of available, no augmentation)
#   TEST    → use real images only (min of available, no augmentation)
#
# Normal = equal halves from pneu-source and TB-source.
# Val/Test are smaller but that is correct — we want clean real images there.

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
    print('Removed old dataset dir.')

for split in ['train', 'val', 'test']:
    for cls in CLASS_NAMES:
        (DATASET_DIR / split / cls).mkdir(parents=True, exist_ok=True)

# ── Compute per-split targets ─────────────────────────────────
TRAIN_TARGET = TARGET_PER_CLASS            # upsample train to this
# val/test: use however many real images exist (balanced to smallest)
val_counts  = [len(npneu_val)//2 + len(ntb_val)//2, len(pneu_val),  len(tb_val)]
test_counts = [len(npneu_test)//2+ len(ntb_test)//2, len(pneu_test), len(tb_test)]
VAL_TARGET  = min(val_counts)
TEST_TARGET = min(test_counts)

print(f'Train target : {TRAIN_TARGET} per class')
print(f'Val target   : {VAL_TARGET}  per class')
print(f'Test target  : {TEST_TARGET} per class')

split_summary = defaultdict(dict)

# ── Helper: copy real images (no augment, no overwrite risk) ──
def copy_real(paths: list, target: int, cls_tag: str, split_tag: str, dst_dir: Path) -> int:
    dst_dir.mkdir(parents=True, exist_ok=True)
    count = 0
    for i, src in enumerate(paths[:target]):
        dst = dst_dir / f'{cls_tag}_{split_tag}_{i:05d}{src.suffix.lower()}'
        try:
            shutil.copy2(src, dst)
            count += 1
        except Exception:
            pass
    return count

print('\nBuilding TRAIN split (with upsampling)...')
# Normal train: NORMAL_FROM_EACH from each source
norm_train_needed_each = TRAIN_TARGET // 2
n_train = upsample_paths(npneu_train, norm_train_needed_each, 'normal', 'pneu',
                          DATASET_DIR / 'train' / 'normal')
n_train += upsample_paths(ntb_train,   norm_train_needed_each, 'normal', 'tb',
                           DATASET_DIR / 'train' / 'normal')
split_summary['train']['normal'] = len(list((DATASET_DIR/'train'/'normal').glob('*')))

p_train = upsample_paths(pneu_train, TRAIN_TARGET, 'pneumonia', 'tr',
                          DATASET_DIR / 'train' / 'pneumonia')
split_summary['train']['pneumonia'] = len(p_train)

t_train = upsample_paths(tb_train, TRAIN_TARGET, 'tuberculosis', 'tr',
                          DATASET_DIR / 'train' / 'tuberculosis')
split_summary['train']['tuberculosis'] = len(t_train)

print('\nBuilding VAL split (real images only)...')
# Normal val: equal from each source
n_val_each = VAL_TARGET // 2
n = copy_real(npneu_val, n_val_each, 'normal', 'pneuvl', DATASET_DIR/'val'/'normal')
n += copy_real(ntb_val,   n_val_each, 'normal', 'tbvl',   DATASET_DIR/'val'/'normal')
split_summary['val']['normal'] = n
split_summary['val']['pneumonia']    = copy_real(pneu_val,  VAL_TARGET,  'pneumonia',    'vl', DATASET_DIR/'val'/'pneumonia')
split_summary['val']['tuberculosis'] = copy_real(tb_val,    VAL_TARGET,  'tuberculosis', 'vl', DATASET_DIR/'val'/'tuberculosis')

print('\nBuilding TEST split (real images only)...')
n_test_each = TEST_TARGET // 2
n = copy_real(npneu_test, n_test_each, 'normal', 'pneute', DATASET_DIR/'test'/'normal')
n += copy_real(ntb_test,   n_test_each, 'normal', 'tbte',   DATASET_DIR/'test'/'normal')
split_summary['test']['normal'] = n
split_summary['test']['pneumonia']    = copy_real(pneu_test,  TEST_TARGET,  'pneumonia',    'te', DATASET_DIR/'test'/'pneumonia')
split_summary['test']['tuberculosis'] = copy_real(tb_test,    TEST_TARGET,  'tuberculosis', 'te', DATASET_DIR/'test'/'tuberculosis')

print('\n── Final counts ────────────────────────────────────────')
for split in ['train', 'val', 'test']:
    for cls in CLASS_NAMES:
        actual = len(list((DATASET_DIR/split/cls).glob('*')))
        split_summary[split][cls] = actual
        print(f'  {split:5s}/{cls:15s}: {actual}')
print('\n✓ Dataset built at:', DATASET_DIR)

Train target : 1200 per class
Val target   : 70  per class
Test target  : 70 per class

Building TRAIN split (with upsampling)...

Building VAL split (real images only)...

Building TEST split (real images only)...

── Final counts ────────────────────────────────────────
  train/normal         : 4063
  train/pneumonia      : 3396
  train/tuberculosis   : 1200
  val  /normal         : 70
  val  /pneumonia      : 70
  val  /tuberculosis   : 70
  test /normal         : 70
  test /pneumonia      : 70
  test /tuberculosis   : 70

✓ Dataset built at: /kaggle/working/lung_dataset


In [ ]:
# ============================================================
# CELL 9 — Data Leakage Verification (MD5-based)
# ============================================================
# Augmented images in train are NEW files with different content
# from their sources, so MD5 won't match. This check confirms
# no ORIGINAL file was accidentally copied into multiple splits.

def get_hashes(split_path: Path) -> set:
    hashes = set()
    for f in split_path.rglob('*'):
        if f.suffix.lower() in IMAGE_EXTENSIONS:
            try:
                hashes.add(md5_hash(f))
            except Exception:
                pass
    return hashes

print('Computing MD5 hashes across splits (takes ~1 min)...')
h_train = get_hashes(DATASET_DIR / 'train')
h_val   = get_hashes(DATASET_DIR / 'val')
h_test  = get_hashes(DATASET_DIR / 'test')

tv = h_train & h_val
tt = h_train & h_test
vt = h_val   & h_test

print(f'train ∩ val  = {len(tv)}  (should be 0)')
print(f'train ∩ test = {len(tt)} (should be 0)')
print(f'val   ∩ test = {len(vt)}  (should be 0)')

if tv or tt or vt:
    print('WARNING: Leakage detected!')
else:
    print('✓ Zero data leakage confirmed.')

In [ ]:
# ============================================================
# CELL 10 — Distribution Visualisation
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ['#4CAF50', '#2196F3', '#FF5722']

for i, split in enumerate(['train', 'val', 'test']):
    counts = [split_summary[split][c] for c in CLASS_NAMES]
    axes[i].bar(CLASS_NAMES, counts, color=colors)
    axes[i].set_title(f'{split.upper()} split')
    axes[i].set_ylabel('Images')
    for j, v in enumerate(counts):
        axes[i].text(j, v + 2, str(v), ha='center', fontsize=9)

plt.suptitle('Class Distribution per Split (3-Class Lung Dataset)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'dataset_distribution.png', dpi=150)
plt.show()
print('Saved.')

In [ ]:
# ============================================================
# CELL 11 — Sample Images
# ============================================================
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle('Sample Images per Class (train)', fontsize=14, fontweight='bold')

for row, cls in enumerate(CLASS_NAMES):
    imgs = [p for p in (DATASET_DIR / 'train' / cls).rglob('*')
            if p.suffix.lower() in IMAGE_EXTENSIONS and not p.name.startswith('._')][:4]
    for col in range(4):
        ax = axes[row][col]
        if col < len(imgs):
            try:
                img = Image.open(imgs[col]).convert('RGB').resize((224, 224))
                ax.imshow(img)
            except Exception:
                ax.text(0.5, 0.5, 'error', ha='center', va='center')
        ax.axis('off')
        if col == 0:
            ax.set_title(cls, fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'sample_images.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# CELL 12 — Transforms & Safe Dataset
# ============================================================
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class SafeImageFolder(Dataset):
    """ImageFolder that skips unreadable/junk images at load time."""
    def __init__(self, root, transform=None):
        from torchvision.datasets import ImageFolder
        base = ImageFolder(root, transform=None)
        self.transform   = transform
        self.classes     = base.classes
        self.class_to_idx = base.class_to_idx
        valid, bad = [], 0
        for path, label in base.samples:
            if Path(path).name.startswith('._'):
                bad += 1; continue
            try:
                with Image.open(path) as img: img.verify()
                valid.append((path, label))
            except Exception:
                bad += 1
        self.samples = valid
        split = Path(root).name
        print(f'  {split:5s}: {len(valid)} valid, {bad} skipped')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

print('Building dataloaders...')
train_ds = SafeImageFolder(str(DATASET_DIR / 'train'), transform=train_transforms)
val_ds   = SafeImageFolder(str(DATASET_DIR / 'val'),   transform=eval_transforms)
test_ds  = SafeImageFolder(str(DATASET_DIR / 'test'),  transform=eval_transforms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'\nClass→index : {train_ds.class_to_idx}')
print(f'Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')

In [ ]:
# ============================================================
# CELL 13 — Custom CNN (MultiNet)
# ============================================================
class MultiNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.1),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.1),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.15),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.15),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(256, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )
    def forward(self, x): return self.classifier(self.features(x))

print('MultiNet defined.')

In [ ]:
# ============================================================
# CELL 14 — Model Factory
# ============================================================
def get_model(name: str) -> nn.Module:
    p = USE_PRETRAINED
    if name == 'multinet':
        return MultiNet(NUM_CLASSES)
    elif name == 'resnet18':
        m = models.resnet18(weights=ResNet18_Weights.DEFAULT if p else None)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES); return m
    elif name == 'resnet34':
        m = models.resnet34(weights=ResNet34_Weights.DEFAULT if p else None)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES); return m
    elif name == 'resnet50':
        m = models.resnet50(weights=ResNet50_Weights.DEFAULT if p else None)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES); return m
    elif name == 'mobilenet_v3_small':
        m = models.mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT if p else None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, NUM_CLASSES); return m
    elif name == 'mobilenet_v3_large':
        m = models.mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT if p else None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, NUM_CLASSES); return m
    elif name == 'efficientnet_b0':
        m = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT if p else None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, NUM_CLASSES); return m
    elif name == 'efficientnet_v2_s':
        m = models.efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT if p else None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, NUM_CLASSES); return m
    elif name == 'vgg16':
        m = models.vgg16(weights=VGG16_Weights.DEFAULT if p else None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, NUM_CLASSES); return m
    elif name == 'densenet121':
        m = models.densenet121(weights=DenseNet121_Weights.DEFAULT if p else None)
        m.classifier = nn.Linear(m.classifier.in_features, NUM_CLASSES); return m
    elif name == 'convnext_tiny':
        m = models.convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT if p else None)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, NUM_CLASSES); return m
    else:
        raise ValueError(f'Unknown model: {name}')

print('Model factory ready.')

In [ ]:
# ============================================================
# CELL 15 — Training & Evaluation Utilities
# ============================================================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward(); optimizer.step()
        with torch.no_grad():
            preds = model(imgs).argmax(1)
        running_loss += loss.item() * imgs.size(0)
        correct += (model(imgs).argmax(1) == labels).sum().item()
        total   += labels.size(0)
    return running_loss / total, correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out  = model(imgs)
            loss = criterion(out, labels)
            preds = out.argmax(1)
            running_loss += loss.item() * imgs.size(0)
            correct += preds.eq(labels).sum().item()
            total   += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return running_loss / total, correct / total, all_preds, all_labels

def count_params(model): return sum(p.numel() for p in model.parameters() if p.requires_grad)

print('Utilities defined.')

In [ ]:
# ============================================================
# CELL 16 — Main Training Loop
# ============================================================
all_results   = []
all_histories = {}

for model_name in MODELS_TO_RUN:
    print(f'\n{"="*60}')
    print(f'  Training: {model_name.upper()}')
    print(f'{"="*60}')

    model     = get_model(model_name).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=2, factor=0.5
    )

    best_val_acc = 0.0
    best_epoch   = 0
    patience_ctr = 0
    save_path    = ARTIFACT_DIR / f'{model_name}_best.pth'
    history      = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}

    start_time = time.time()

    for epoch in range(1, EPOCHS + 1):
        t_loss, t_acc           = train_one_epoch(model, train_loader, criterion, optimizer, device)
        v_loss, v_acc, _, _     = evaluate(model, val_loader, criterion, device)
        scheduler.step(v_acc)

        history['train_loss'].append(t_loss)
        history['train_acc'].append(t_acc)
        history['val_loss'].append(v_loss)
        history['val_acc'].append(v_acc)

        print(f'Epoch {epoch:02d}/{EPOCHS}  '
              f'Train Loss:{t_loss:.4f} Acc:{t_acc:.4f}  |  '
              f'Val Loss:{v_loss:.4f} Acc:{v_acc:.4f}')

        if v_acc > best_val_acc:
            best_val_acc = v_acc; best_epoch = epoch; patience_ctr = 0
            torch.save(model.state_dict(), save_path)
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                print(f'Early stopping at epoch {epoch}.'); break

    # ── Test evaluation ──────────────────────────────────────
    model.load_state_dict(torch.load(save_path, map_location=device))
    _, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion, device)
    prec, rec, f1, _ = precision_recall_fscore_support(
        test_labels, test_preds, average='weighted', zero_division=0
    )
    elapsed = time.time() - start_time

    all_results.append({
        'model': model_name, 'best_epoch': best_epoch,
        'best_val_acc': round(best_val_acc, 5), 'test_acc': round(test_acc, 6),
        'precision_weighted': round(prec, 6), 'recall_weighted': round(rec, 6),
        'f1_weighted': round(f1, 6), 'parameters': count_params(model),
        'train_time_s': round(elapsed, 1),
    })
    all_histories[model_name] = history

    with open(ARTIFACT_DIR / f'{model_name}_report.txt', 'w') as f:
        f.write(classification_report(test_labels, test_preds, target_names=CLASS_NAMES, zero_division=0))

    print(f'\nTest Acc:{test_acc:.4f}  F1:{f1:.4f}  Params:{count_params(model):,}  Time:{elapsed:.0f}s')
    del model; torch.cuda.empty_cache(); gc.collect()

print('\n✓ All models trained.')

In [ ]:
# ============================================================
# CELL 17 — Results Table
# ============================================================
df_results = pd.DataFrame(all_results).sort_values('test_acc', ascending=False).reset_index(drop=True)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)
print(df_results.to_string(index=False))
df_results.to_csv(ARTIFACT_DIR / 'model_comparison.csv', index=False)
print('\nSaved model_comparison.csv')

In [ ]:
# ============================================================
# CELL 18 — Training Curves (all models)
# ============================================================
def plot_history(history, model_name, save_dir):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ep = range(1, len(history['train_acc']) + 1)
    ax1.plot(ep, history['train_acc'], label='Train', marker='o', markersize=3)
    ax1.plot(ep, history['val_acc'],   label='Val',   marker='s', markersize=3)
    ax1.set_title(f'{model_name} — Accuracy'); ax1.set_xlabel('Epoch')
    ax1.legend(); ax1.grid(True, alpha=0.3)
    ax2.plot(ep, history['train_loss'], label='Train', marker='o', markersize=3)
    ax2.plot(ep, history['val_loss'],   label='Val',   marker='s', markersize=3)
    ax2.set_title(f'{model_name} — Loss'); ax2.set_xlabel('Epoch')
    ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_dir / f'{model_name}_curves.png', dpi=120)
    plt.show()

for mname, hist in all_histories.items():
    plot_history(hist, mname, ARTIFACT_DIR)

In [ ]:
# ============================================================
# CELL 19 — Confusion Matrices (Top 3 Models)
# ============================================================
top3 = df_results['model'].head(3).tolist()
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, mname in enumerate(top3):
    model = get_model(mname).to(device)
    model.load_state_dict(torch.load(ARTIFACT_DIR / f'{mname}_best.pth', map_location=device))
    _, _, preds, labels = evaluate(model, test_loader, nn.CrossEntropyLoss(), device)
    cm = confusion_matrix(labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[idx])
    axes[idx].set_title(f'{mname}\nTest Acc:{accuracy_score(labels, preds):.4f}')
    axes[idx].set_xlabel('Predicted'); axes[idx].set_ylabel('True')
    del model; torch.cuda.empty_cache(); gc.collect()

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'top3_confusion_matrices.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# CELL 20 — Model Comparison Bar Chart
# ============================================================
fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(df_results['model'], df_results['test_acc'], color='steelblue', edgecolor='white')
ax.set_xlabel('Test Accuracy', fontsize=12)
ax.set_title('Model Comparison — Test Accuracy (3-Class Lung Disease)', fontsize=13, fontweight='bold')
ax.set_xlim(0, 1.05)
for bar, acc in zip(bars, df_results['test_acc']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{acc:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'model_comparison_bar.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# CELL 21 — Ensemble (Top-2) + TTA
# ============================================================
best_name   = df_results.iloc[0]['model']
second_name = df_results.iloc[1]['model']
print(f'Ensemble: {best_name} + {second_name}')

def load_best(name):
    m = get_model(name).to(device)
    m.load_state_dict(torch.load(ARTIFACT_DIR / f'{name}_best.pth', map_location=device))
    m.eval(); return m

model1 = load_best(best_name)
model2 = load_best(second_name)

all_ens_preds, all_ens_labels = [], []

for imgs, labels in test_loader:
    imgs   = imgs.to(device)
    labels = labels.to(device)
    imgs_flip = torch.flip(imgs, dims=[3])
    with torch.no_grad():
        prob = (F.softmax(model1(imgs),      dim=1) +
                F.softmax(model1(imgs_flip),  dim=1) +
                F.softmax(model2(imgs),       dim=1) +
                F.softmax(model2(imgs_flip),  dim=1)) / 4.0
    preds = prob.argmax(dim=1)
    all_ens_preds.extend(preds.cpu().numpy())
    all_ens_labels.extend(labels.cpu().numpy())

ens_acc = accuracy_score(all_ens_labels, all_ens_preds)
ens_prec, ens_rec, ens_f1, _ = precision_recall_fscore_support(
    all_ens_labels, all_ens_preds, average='weighted', zero_division=0
)
print(f'\nEnsemble+TTA  Acc:{ens_acc:.4f}  Prec:{ens_prec:.4f}  Rec:{ens_rec:.4f}  F1:{ens_f1:.4f}')

ensemble_row = pd.DataFrame([{
    'model': f'Ensemble_{best_name}+{second_name}_TTA',
    'best_epoch':'N/A', 'best_val_acc':'N/A',
    'test_acc': round(ens_acc, 6),
    'precision_weighted': round(ens_prec, 6),
    'recall_weighted': round(ens_rec, 6),
    'f1_weighted': round(ens_f1, 6),
    'parameters': count_params(model1)+count_params(model2),
    'train_time_s':'N/A',
}])
df_final = pd.concat([df_results, ensemble_row], ignore_index=True)
df_final = df_final.sort_values('test_acc', ascending=False).reset_index(drop=True)
df_final.to_csv(ARTIFACT_DIR / 'final_model_comparison.csv', index=False)

print('\n--- FINAL LEADERBOARD ---')
print(df_final[['model','test_acc','f1_weighted','parameters','train_time_s']].to_string(index=False))

del model1, model2; torch.cuda.empty_cache()

In [ ]:
# ============================================================
# CELL 22 — Ensemble Confusion Matrix
# ============================================================
fig, ax = plt.subplots(figsize=(6, 5))
cm_ens = confusion_matrix(all_ens_labels, all_ens_preds)
sns.heatmap(cm_ens, annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title(f'Ensemble+TTA — Test Acc:{ens_acc:.4f}')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'ensemble_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# CELL 23 — Final Summary
# ============================================================
total_imgs = len(train_ds) + len(val_ds) + len(test_ds)
print('=' * 65)
print('  LUNG DISEASE CLASSIFICATION — FINAL SUMMARY')
print('=' * 65)
print(f'  Classes         : {CLASS_NAMES}')
print(f'  Total images    : {total_imgs}')
print(f'  Train / Val / Test : {len(train_ds)} / {len(val_ds)} / {len(test_ds)}')
print(f'  Target per class: {TARGET_PER_CLASS} (train, upsampled)')
print(f'  Normal source   : {NORMAL_FROM_EACH} from pneu-dataset + {NORMAL_FROM_EACH} from TB-dataset')
print(f'  Image size      : {IMG_SIZE}x{IMG_SIZE}  Batch:{BATCH_SIZE}  Epochs:{EPOCHS}')
print(f'  Device          : {device} ({gpu_name})')
print()
print(df_final[['model','test_acc','f1_weighted']].to_string(index=False))
print()
print(f'  Best model      : {df_results.iloc[0]["model"]} ({df_results.iloc[0]["test_acc"]:.4f})')
print(f'  Ensemble+TTA    : {ens_acc:.4f}')
print('=' * 65)
print('  Artifacts saved to:', ARTIFACT_DIR)

In [ ]:
# ============================================================
# CELL 24 — Grad-CAM Predictions (ResNet18)
# ============================================================
# Install the grad-cam library if you haven't already
!pip install -q grad-cam

import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

print("Loading ResNet18 for Grad-CAM...")

# 1. Load the best model (ResNet18)
model_name = 'resnet18'
model = get_model(model_name).to(device)
model.load_state_dict(torch.load(ARTIFACT_DIR / f'{model_name}_best.pth', map_location=device))
model.eval()

# 2. Define the target layer for ResNet18
# The last convolutional layer in ResNet18 is inside layer4
target_layers = [model.layer4[-1]]

# 3. Initialize Grad-CAM
cam = GradCAM(model=model, target_layers=target_layers)

# 4. Select 10 random images from the test dataset
# test_ds.samples contains tuples of (file_path, label_index)
random.seed(SEED) # Keep it reproducible
sample_indices = random.sample(range(len(test_ds)), 10)

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
fig.suptitle('Grad-CAM Visualisations (ResNet18)', fontsize=16, fontweight='bold')
axes = axes.flatten()

for i, idx in enumerate(sample_indices):
    path, true_label_idx = test_ds.samples[idx]
    true_label = CLASS_NAMES[true_label_idx]
    
    # Load the raw image for visualization
    raw_img = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    rgb_img = np.float32(raw_img) / 255.0 # Normalize to [0, 1] for show_cam_on_image
    
    # Prepare the tensor using your eval_transforms (from Cell 12)
    input_tensor = eval_transforms(raw_img).unsqueeze(0).to(device)
    
    # Run Inference
    with torch.no_grad():
        out = model(input_tensor)
        pred_label_idx = out.argmax(1).item()
        pred_label = CLASS_NAMES[pred_label_idx]
        
    # Generate the CAM for the predicted class
    targets = [ClassifierOutputTarget(pred_label_idx)]
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
    grayscale_cam = grayscale_cam[0, :] # Extract the single batch item
    
    # Overlay heatmap on the original image
    cam_image = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
    
    # Plotting
    ax = axes[i]
    ax.imshow(cam_image)
    
    # Color text green if correct, red if incorrect
    title_color = 'darkgreen' if true_label == pred_label else 'darkred'
    ax.set_title(f"True: {true_label}\nPred: {pred_label}", 
                 fontsize=11, fontweight='bold', color=title_color)
    ax.axis('off')

plt.tight_layout()
plt.savefig(ARTIFACT_DIR / 'gradcam_predictions.png', dpi=150)
plt.show()

print(f"Grad-CAM visualisations saved to: {ARTIFACT_DIR / 'gradcam_predictions.png'}")